<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 3.4 — Sensitivity Analysis: What Is One More Unit Worth?

Fitting the model is the easy part. **Interrogating** it is the job.

A **binding** constraint is a bottleneck: it is the thing stopping you from doing better. So the natural
question is: *if I could loosen it a little, how much better would I do?* The answer is called the
**shadow price**.

You do not need anything fancy to find it. **Put the model in a for loop, change one number each time,
and look at how the answer moves.**

Two examples:

1. **Flair's Furniture:** what is one more hour on the carpentry machine worth?
2. **McDonald's:** what is the shadow price of salt?

All of the code is already here. Run each cell and read what it prints.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

# Part 1 · What is one more carpentry machine hour worth?

Think of Flair's carpentry hours as hours the **carpentry machine** can run each month: 2,400 of them.
Here is the model from 3.1 once more, with the binding check.

In [ ]:
# Flair's Furniture, exactly as in 3.1
model = ConcreteModel()
model.tables = Var(domain=NonNegativeReals)
model.chairs = Var(domain=NonNegativeReals)
model.profit = Objective(expr = 7*model.tables + 5*model.chairs, sense = maximize)
model.constraint1 = Constraint(expr = 3*model.tables + 4*model.chairs <= 2400) # carpentry machine hours
model.constraint2 = Constraint(expr = 2*model.tables + 1*model.chairs <= 1000) # painting hours
model.constraint3 = Constraint(expr = model.chairs <= 450) # chair limit
model.constraint4 = Constraint(expr = model.tables >= 100) # table minimum
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

print("Profit = ", model.profit(), " USD  (", model.tables(), "tables,", model.chairs(), "chairs )")
print()
names = ["Carpentry machine hours", "Painting hours", "Chair limit", "Table minimum"]
constraints = [model.constraint1, model.constraint2, model.constraint3, model.constraint4]
for i in range(len(constraints)):
    c = constraints[i]
    slack = min(c.uslack(), c.lslack())
    if slack < 0.000001:
        print(names[i], ": slack =", slack, " <-- BINDING (a bottleneck!)")
    else:
        print(names[i], ": slack =", slack)

**All 2,400 carpentry hours are used up.** The machine is a bottleneck. If it could run a few more
hours a month, Flair could make more furniture and more money. **But how much more, per hour?**

Let's find out. We solve the model again and again, **changing only the carpentry hours**: from 1,200
to 3,200, in steps of 100. The base case (2,400 hours) is in the middle, so we can see what happens
both ways.

In [ ]:
base_hours = 2400                          # the real number of carpentry machine hours
myHours = np.arange(1200, 3300, 100)       # the hours we will try
print(myHours)

The loop is the model from above with **one change**: the `2400` became `a`.

In [ ]:
# store the results
myResults = pd.DataFrame() # empty place to store results

for a in myHours:
  # declare the model
  model = ConcreteModel()

  # declare decision variables
  model.tables = Var(domain=NonNegativeReals)
  model.chairs = Var(domain=NonNegativeReals)

  # declare objective
  model.profit = Objective(
                        expr = 7*model.tables + 5*model.chairs,
                        sense = maximize)

  # declare constraints - carpentry hours are now 'a', not 2400
  model.constraint1 = Constraint(expr = 3*model.tables + 4*model.chairs <= a) # carpentry machine hours
  model.constraint2 = Constraint(expr = 2*model.tables + 1*model.chairs <= 1000) # painting hours
  model.constraint3 = Constraint(expr = model.chairs <= 450) # chair limit
  model.constraint4 = Constraint(expr = model.tables >= 100) # table minimum

  # solve it (no .write() here, or you get 21 printouts!)
  SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

  # results
  myX = pd.DataFrame([a, model.profit(), model.tables(), model.chairs()])
  myX = myX.T # transpose

  # store the results
  myResults = pd.concat([myResults, myX])

# Change the Column names
myResults = myResults.rename({0:"Carpentry Hours", 1:"Profit", 2:"Tables", 3:"Chairs"}, axis='columns')
myResults.reset_index(drop=True, inplace=True)

Now the shadow price. Each row has 100 more hours than the row above, so:

**value of one more hour = (profit in this row − profit in the row above) ÷ 100**

The table marks the base case so you can find it.

In [ ]:
# what is ONE more carpentry machine hour worth?
myResults['Value of one more hour ($)'] = myResults['Profit'].diff() / myResults['Carpentry Hours'].diff()

# mark the base case
myResults['note'] = ''
myResults.loc[myResults['Carpentry Hours'] == base_hours, 'note'] = '<-- base (2,400 hours)'

myResults.round(2)

In [ ]:
# picture 1: the product mix - what does Flair make at each number of hours?
tmp = myResults.drop(['Profit', 'Value of one more hour ($)', 'note'], axis=1)
tmp.plot(x='Carpentry Hours')
axvline(base_hours, color='red', linewidth=3, label='base: 2,400 hours')
legend()
title('Product mix')
show()

# picture 2: what is one more hour worth?
myResults.plot(x='Carpentry Hours', y='Value of one more hour ($)', color='darkorange')
axvline(base_hours, color='red', linewidth=3, label='base: 2,400 hours')
legend()
title('What is one more carpentry machine hour worth?')
show()

**Read the second picture.** At the red line (2,400 hours, where Flair is today), one more machine hour
is worth **\$0.60** of profit. That is the **shadow price** of carpentry.

Move right and it stays \$0.60 for a while, then drops to **\$0** at about 2,600 hours. Why? The
first picture shows it: by then Flair is making the most chairs it is allowed (450). More machine time
is useless, because the bottleneck has moved to the chair limit. Carpentry hours start out like
**diamonds** and turn into **sand**.

(Move left instead, below 1,500 hours, and an hour is worth \$2.33: with very little machine time,
Flair makes only tables.)

**The business question.** Suppose running the carpentry machine costs Flair **\$0.50 an hour**
(electricity, wear and tear). Should Flair run it longer?

- **Up to about 2,600 hours: yes.** Each extra hour costs \$0.50 and earns \$0.60, so Flair comes out
  10 cents ahead per hour.
- **Past about 2,600 hours: no.** Each extra hour still costs \$0.50 but now earns \$0.

**On your own:** what is one more **painting** hour worth? In the loop, change the painting constraint
to `<= a` and the carpentry one back to `<= 2400`, and try `myHours = np.arange(500, 2000, 100)` with
`base_hours = 1000`.

# Part 2 · The shadow price of salt at McDonald's

Same idea, on a bigger model. In 3.3 the class rules allowed each player **at most 7,000 mg of sodium**.
Cheap fast food is salty, so a sodium limit can cost money.

The sodium rule is a **maximum**: each player may have at most so many milligrams of sodium a day.
When that rule is binding, it is costing the team money, because the cheapest foods are salty.

> **The shadow price of salt** is the answer to one question:
> **if the sodium limit were 100 mg higher, how many dollars per player would the order get cheaper?**

Read it the other way and it is just as useful: **what does it cost, per player, to cut 100 mg of
sodium?**

We find it exactly the way we found the value of a carpentry hour. Solve the model at one sodium
limit, solve it again with the limit 100 mg higher, and subtract.

Three things to be clear about:

1. **The units are dollars per player, per 100 mg of sodium.** For the whole team, multiply by the team size.
2. **It is not one number.** It depends on where the limit is. A tight limit has a high shadow price;
   a loose limit has a shadow price of zero.
3. **Zero means the rule is not binding.** Above some limit the cheapest order does not even use all
   the sodium it is allowed, so allowing more saves nothing. Diamonds to sand.

In [ ]:
# read the McDonald's data (same as notebook 3.3)
data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/menus/'
menu_file = 'mcdonalds_menu.csv'
menu_df = pd.read_csv(data_folder + menu_file)
rules_df = pd.read_csv(data_folder + 'team_rules.csv')

items = list(menu_df['item'])
nutrient_columns = list(menu_df.columns[3:])

price = {}
nutrition = {}
for i in range(len(menu_df)):
    item = menu_df.loc[i, 'item']
    price[item] = float(menu_df.loc[i, 'price'])
    nutrition[item] = {}
    for nutrient in nutrient_columns:
        nutrition[item][nutrient] = float(menu_df.loc[i, nutrient])

player_min = {}
player_max = {}
for i in range(len(rules_df)):
    nutrient = rules_df.loc[i, 'nutrient']
    if rules_df.loc[i, 'rule'] == 'min':
        player_min[nutrient] = int(rules_df.loc[i, 'per_player'])
    else:
        player_max[nutrient] = int(rules_df.loc[i, 'per_player'])

print("minimums per player:", player_min)
print("maximums per player:", player_max)

We try every sodium limit from 4,000 mg to 7,500 mg, 100 mg at a time. The base case is the class
limit, **7,000 mg**. The loop is the team order model from 3.3 with **one change**: the sodium limit
is `a`.

In [ ]:
# ---------- THE SCENARIO (change things here) ----------
base_limit = 7000                        # the sodium limit we actually used in class (the base case)
myRange = np.arange(4000, 7600, 100)    # the sodium limits to try, in steps of 100 mg
team_size = 53
max_of_one_item = 4

In [ ]:
# ---------- THE LOOP (you never change this cell) ----------
saltResults = pd.DataFrame()

for a in myRange:
    low = dict(player_min)
    high = dict(player_max)

    # the ONE thing we change each time through the loop: the sodium limit
    high['sodium'] = int(a)

    # the model (same as the team order notebook)
    model = ConcreteModel()
    model.x = Var(items, domain=NonNegativeReals)
    obj_expr = 0
    for item in items:
        obj_expr += price[item] * model.x[item]
    model.cost = Objective(expr = obj_expr, sense = minimize)
    model.constraints = ConstraintList()
    for nutrient in low:
        nutrient_expr = 0
        for item in items:
            nutrient_expr += nutrition[item][nutrient] * model.x[item]
        model.constraints.add(nutrient_expr >= low[nutrient] * team_size)
    for nutrient in high:
        nutrient_expr = 0
        for item in items:
            nutrient_expr += nutrition[item][nutrient] * model.x[item]
        model.constraints.add(nutrient_expr <= high[nutrient] * team_size)
    for item in items:
        model.constraints.add(model.x[item] <= max_of_one_item * team_size)

    results = SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

    # store the cost per player (if there is no feasible order, store a blank)
    if results.solver.termination_condition == TerminationCondition.optimal:
        cost_per_player = model.cost() / team_size
    else:
        cost_per_player = np.nan
    myX = pd.DataFrame([a, cost_per_player])
    myX = myX.T
    saltResults = pd.concat([saltResults, myX])

saltResults = saltResults.rename({0:"Sodium limit (mg)", 1:"Cost per player ($)"}, axis='columns')
saltResults.reset_index(drop=True, inplace=True)
saltResults

## The shadow price, one subtraction at a time

Each row of the table is 100 mg more sodium than the row above it. So:

**shadow price of salt = (cost in the row above) − (cost in this row)**

That is how many dollars per player we save when the limit goes up by 100 mg. The next cell does the
subtraction for every row. (`diff()` gives this row minus the row above, which is negative because the
cost is falling, so we flip the sign.)

In [ ]:
# dollars saved per player when the sodium limit goes up by 100 mg
saltResults['Shadow price of salt ($ per 100 mg)'] = -saltResults['Cost per player ($)'].diff()
saltResults['Shadow price of salt ($ per 100 mg)'] = saltResults['Shadow price of salt ($ per 100 mg)'].round(3) + 0.0    # + 0.0 turns -0.0 into 0.0

# mark the base case
saltResults['note'] = ''
saltResults.loc[saltResults['Sodium limit (mg)'] == base_limit, 'note'] = '<-- base (class limit)'

# show the whole table
pd.set_option('display.max_rows', 100)
saltResults.round(3)

In [ ]:
# picture 1: the bill
saltResults.plot(x='Sodium limit (mg)', y='Cost per player ($)')
axvline(base_limit, color='red', linewidth=3, label='base: the class limit')
legend()
title('Cost per player at each sodium limit')
show()

# picture 2: the shadow price
saltResults.plot(x='Sodium limit (mg)', y='Shadow price of salt ($ per 100 mg)', color='darkorange')
axvline(base_limit, color='red', linewidth=3, label='base: the class limit')
legend()
title('The shadow price of salt')
show()

## How to read it

**Start at the red line.** That is the class limit, 7,000 mg, where we solved the model in 3.3. The
cheapest order there only has about 6,260 mg of sodium per player, so the shadow price at the red line is
**\$0**: at the class limit, salt is not costing the team anything.

Now read the second picture from right to left, tightening the limit:

- **Flat at \$0 on the right.** The limit is loose. The cheapest order does not use all the sodium it
  is allowed, so the rule is **not binding** and 100 mg more is worth nothing.
- **Then it steps up.** Now the rule is binding. Every 100 mg you take away forces the order toward
  less salty, more expensive food. The height of the step is what each 100 mg costs per player.
- **It climbs as the limit gets tighter.** The cheap low-sodium options run out, so each cut hurts
  more than the last one.
- **Then the line stops.** Blanks (`NaN`) in the table mean **infeasible**: no order from this menu can
  meet every rule at that sodium limit, at any price.

**Saying it out loud.** If the shadow price at a 5,500 mg limit is \$0.33, then: *"At a 5,500 mg
limit, letting each player have 100 mg more sodium would save 33 cents per player, or about \$17.50 for
a team of 53. Cutting 100 mg would cost that much."*

**Your turn. Use the table and the two pictures:**

1. Start at the red line and move left. At what sodium limit does the shadow price **first rise above
   \$0**? What does that tell you about the class limit of 7,000 mg?
2. What is the shadow price of salt at a 5,500 mg limit? Say it in a full sentence, with units.
3. What is the lowest sodium limit this menu can meet?
4. The team doctor wants to go from 6,000 mg down to 5,000 mg. What does that cost per player? For the
   whole team of 53?

## Bottom line

- A **binding constraint** is a bottleneck. Its **shadow price** is what one more unit is worth.
- To find it: **put the model in a for loop**, change the right-hand side, and look at the difference
  from row to row. Draw a red line at the base case so you know where you are.
- Shadow prices do not last forever. Scarce things are valuable; once a resource is no longer scarce,
  it is worth nothing and the bottleneck is somewhere else. **Diamonds to sand.**
- Compare the shadow price with what it costs to loosen the constraint (\$0.50 a machine hour, a
  doctor's sodium limit) to make a business decision.

**Before you leave:** File → Save a copy in GitHub.